# 🏥 Chest X-Ray Pneumonia Detection - High Accuracy Pipeline

This notebook provides a complete walk-through of the training process for detecting pneumonia using the **RSNA Pneumonia Detection Challenge** dataset. 

### ⭐ Super-Medical-Cleaner Included:
- **Watermark Removal**: Automatic text/marker scrubbing using OpenCV **Inpainting** (Telea).
- **Feature Enhancement**: **CLAHE** (Contrast Limited Adaptive Histogram Equalization) to normalize lighting and emphasize lung opacities.

### Key Improvements for Accuracy:
- **Architecture**: Switched from MobileNetV2 to **EfficientNetV2-B0**.
- **Resolution**: Increased from 160x160 to **224x224**.
- **Dataset**: Balanced subset of **12,000 images** (6,000 per class).
- **Learning Strategy**: Fine-tuning the top 50 layers with a Cosine Decay learning rate schedule.

## ⚙️ 1. Setup & Configuration
First, we define our paths and hyperparameters.

In [ ]:
import os, sys, logging, random, shutil, subprocess, time
from pathlib import Path
import pandas as pd
import numpy as np
from concurrent.futures import ThreadPoolExecutor, as_completed
try:
    import pydicom
    from pydicom.pixel_data_handlers.util import apply_voi_lut
except ImportError:
    print("pydicom not found, installing...")
    subprocess.run([sys.executable, "-m", "pip", "install", "pydicom pylibjpeg pylibjpeg-libjpeg pylibjpeg-openjpeg"])
    import pydicom
    from pydicom.pixel_data_handlers.util import apply_voi_lut

try:
    import cv2
except ImportError:
    print("opencv not found, installing...")
    subprocess.run([sys.executable, "-m", "pip", "install", "opencv-python"])
    import cv2

try:
    import scipy
    from scipy import ndimage
except ImportError:
    print("scipy not found, installing stable version...")
    subprocess.run([sys.executable, "-m", "pip", "install", "scipy==1.11.4 numpy==1.26.4"])
    from scipy import ndimage

from PIL import Image

# Paths
ROOT = Path(os.getcwd())
RSNA_DIR = ROOT / "rsna-pneumonia-detection-challenge"
RAW_DATA_DIR = ROOT / "data" / "raw" / "chest_xray"
PROC_DATA_DIR = ROOT / "data" / "processed"
MODELS_DIR = ROOT / "models"

MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Config
IMG_SIZE = 224
MAX_PER_CLASS = 6000
BATCH_SIZE = 32
EPOCHS = 20
UNFREEZE_LAYERS = 50
LEARNING_RATE = 4e-4

print(f"Setup Complete! Image Size: {IMG_SIZE}, Cap: {MAX_PER_CLASS}/class")

## 🔄 2. Step 1: DICOM to JPEG Preprocessing (with Cleaning)
We convert `.dcm` files to cleaned, enhanced `.jpg` images.

In [ ]:
def clean_and_enhance_image(img_array):
    """Detects and removes text watermarks and applies CLAHE enhancement."""
    try:
        # 1. CLAHE Enhancement
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        img_enhanced = clahe.apply(img_array)
        
        # 2. Watermark removal (Inpainting)
        # Burnt-in text is usually white (intensity ~255)
        _, mask = cv2.threshold(img_enhanced, 250, 255, cv2.THRESH_BINARY)
        mask = cv2.dilate(mask, np.ones((5, 5), np.uint8), iterations=1)
        
        # 3. Heal the image textures
        return cv2.inpaint(img_enhanced, mask, 3, cv2.INPAINT_TELEA)
    except Exception: 
        return img_array

def dcm_to_jpeg(dcm_path, jpeg_path, quality=95):
    try:
        ds = pydicom.dcmread(str(dcm_path))
        try: img_array = apply_voi_lut(ds.pixel_array, ds)
        except Exception: img_array = ds.pixel_array
        
        img_array = img_array.astype(np.float32)
        img_min, img_max = img_array.min(), img_array.max()
        if img_max > img_min: img_array = (img_array - img_min) / (img_max - img_min) * 255.0
        img_array = img_array.astype(np.uint8)
        
        # --- APPLY ADVANCED CLEANING ---
        img_array = clean_and_enhance_image(img_array)
        
        pil_img = Image.fromarray(img_array)
        if pil_img.mode != "RGB": pil_img = pil_img.convert("RGB")
        pil_img.save(str(jpeg_path), "JPEG", quality=quality)
        return True
    except Exception: return False

def run_rsna_prep():
    labels_csv = RSNA_DIR / "stage_2_train_labels.csv"
    images_dir = RSNA_DIR / "stage_2_train_images"
    if not labels_csv.exists(): return False
    
    df = pd.read_csv(labels_csv)
    patient_labels = df.groupby("patientId")["Target"].max().reset_index()
    patient_labels.columns = ["patientId", "label"]

    for cls in ("normal", "pneumonia"): (RAW_DATA_DIR / "train" / cls).mkdir(parents=True, exist_ok=True)

    tasks = []
    for _, row in patient_labels.iterrows():
        pid, cls = row["patientId"], "pneumonia" if row["label"] == 1 else "normal"
        src, dst = images_dir / f"{pid}.dcm", RAW_DATA_DIR / "train" / cls / f"{pid}.jpg"
        if src.exists() and not dst.exists(): tasks.append((src, dst))

    if not tasks:
        print("All images already converted!")
        return True

    print(f"Processing {len(tasks)} images with Super-Medical-Cleaner...")
    with ThreadPoolExecutor(max_workers=4) as pool: pool.map(lambda x: dcm_to_jpeg(*x), tasks)
    print("Step 1 (Advanced) Complete!")
    return True

run_rsna_prep()

## 📊 3. Step 2: Balanced Data Splitting
We create a stratified split of our images into Training, Validation, and Test sets.

In [ ]:
def build_fast_split():
    # If data already exists and is balanced, skip
    if (PROC_DATA_DIR / "train" / "normal").exists():
        normal_count = len(list((PROC_DATA_DIR / "train" / "normal").glob("*.jpg")))
        if normal_count >= int(MAX_PER_CLASS * 0.7):
            print("Dataset already split and balanced!")
            return

    # Robust Windows cleanup
    if PROC_DATA_DIR.exists():
        try:
            shutil.rmtree(PROC_DATA_DIR)
        except OSError:
            subprocess.run(["powershell", "-Command", f"Remove-Item -Path '{PROC_DATA_DIR}' -Recurse -Force -ErrorAction SilentlyContinue"])


In [ ]:
    # Completion of build_fast_split (re-creating directories and splitting)
    for split in ("train", "val", "test"):
        for cls in ("normal", "pneumonia"):
            (PROC_DATA_DIR / split / cls).mkdir(parents=True, exist_ok=True)

    raw_normal = sorted((RAW_DATA_DIR / "train" / "normal").glob("*.jpg"))
    raw_pneumo = sorted((RAW_DATA_DIR / "train" / "pneumonia").glob("*.jpg"))

    random.seed(42)
    random.shuffle(raw_normal)
    random.shuffle(raw_pneumo)

    cap = min(MAX_PER_CLASS, len(raw_normal), len(raw_pneumo))
    raw_normal, raw_pneumo = raw_normal[:cap], raw_pneumo[:cap]

    def split_list(lst): 
        n = len(lst)
        t, v = int(n*0.7), int(n*0.2)
        return lst[:t], lst[t:t+v], lst[t+v:]

    for imgs, label in [(raw_normal, "normal"), (raw_pneumo, "pneumonia")]:
        tr, va, te = split_list(imgs)
        for f, s in [(tr, "train"), (va, "val"), (te, "test")]:
            dest = PROC_DATA_DIR / s / label
            for img in f: shutil.copy2(img, dest / img.name)
    print("Step 2 Complete!")

build_fast_split()

## 🚀 4. Step 3: Model Architecture & Training
We use EfficientNetV2-B0 with partial unfreezing and mixed precision for optimal learning.

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetV2B0
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, BatchNormalization, Input
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.optimizers.schedules import CosineDecay
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

print("TensorFlow Version:", tf.__version__)

# Enable Mixed Precision if GPU is present
gpus = tf.config.list_physical_devices('GPU')
if gpus: 
    print("GPU detected! Enabling Mixed Precision.")
    tf.keras.mixed_precision.set_global_policy('mixed_float16')

# Data Loaders
datagen = ImageDataGenerator(rescale=1./255, rotation_range=10, horizontal_flip=True)
val_datagen = ImageDataGenerator(rescale=1./255)

train_gen = datagen.flow_from_directory(PROC_DATA_DIR / "train", target_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE, class_mode="binary")
val_gen = val_datagen.flow_from_directory(PROC_DATA_DIR / "val", target_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE, class_mode="binary")

# Model
inp = Input(shape=(IMG_SIZE, IMG_SIZE, 3))
base = EfficientNetV2B0(weights="imagenet", include_top=False, input_tensor=inp)
for layer in base.layers[:-UNFREEZE_LAYERS]: layer.trainable = False

x = GlobalAveragePooling2D()(base.output)
x = BatchNormalization()(x)
x = Dense(256, activation="relu")(x)
x = Dropout(0.3)(x)
out = Dense(1, activation="sigmoid", dtype="float32")(x)
model = Model(inp, out)

# Optimizer
lr_sched = CosineDecay(LEARNING_RATE, decay_steps=len(train_gen)*EPOCHS)
model.compile(optimizer=Adam(lr_sched), loss="binary_crossentropy", metrics=["accuracy"])

# Callbacks
history = model.fit(
    train_gen, epochs=EPOCHS, validation_data=val_gen, 
    callbacks=[EarlyStopping(patience=5, restore_best_weights=True)]
)

## 📈 5. Step 4: Results & Evaluation
Finally, we evaluate on the test set and plot our confusion matrix.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

test_gen = val_datagen.flow_from_directory(PROC_DATA_DIR / "test", target_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE, class_mode="binary", shuffle=False)
preds = (model.predict(test_gen) > 0.5).astype(int)

print(classification_report(test_gen.classes, preds, target_names=["Normal", "Pneumonia"]))

cm = confusion_matrix(test_gen.classes, preds)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("Confusion Matrix")